In [ ]:
from math import ceil

import matplotlib.pyplot as plt
import numpy as np
from numpy import pi
from scipy.interpolate import pchip_interpolate
from scipy.optimize import fsolve

from igs_ga.optimization.modo import Preferendus

In [ ]:
# Constants for the problem

# Set constants for objective functions
max_t = 3800
n_anchors = 108

constants = {
    'NC': 9,
    'NQ': 1,
    'W_steel': 78.5,  # kN/m3
    'W_water': 10.25,  # kN/m3
    'W_concrete': 25,  # kN/m3
}

# Set characteristics of the different ship types
ship_options = {
    'OCV small': {
        'day_rate': 47000,  # €/day
        'deck_space': 8,  # max anchors on deck
        'max_available': 3, # how many of this type are available for the project
        'CO2_emission': 30,  # tonnes per day
        'chance': 0.7
    },
    'OCV big': {
        'day_rate': 55000,
        'deck_space': 12,
        'max_available': 2,
        'CO2_emission': 40,  # tonnes per day
        'chance': 0.8
    },
    'Barge': {
        'day_rate': 35000,
        'deck_space': 16,
        'max_available': 2,
        'CO2_emission': 35,  # tonnes per day
        'chance': 0.5
    }
}

soil_data = {
    'type': 'clay',
    'su': 60,  # kPa (Undrained shear strength)
    'a_i': 0.64, # - (Coefficient of shaft friction)
    'a_o': 0.64,
    'sat_weight': 9,  # kN/m3 (Submerged unit weight)
}

mooring_data = {
    'type': 'catenary',
    'line type': 'chain',
    'd': 0.24,  # m (Nominal chain diameter)
    'mu': 0.25,  # - (Coefficient of seabed friction)
    'AWB': 2.5  # - (Active bearing area coefficient)
}

# set variables for
time_installation = 1  # time it takes to install one anchor
time_bunkering = [1.5, 2, 2.5]  # time it takes to get a new set of anchors onboard


In [ ]:
def objective_time(ocv_s, ocv_l, barge):
    """Function to calculate the project duration

    :param ocv_s: number of small offshore construction vessels
    :param ocv_l: number of large offshore construction vessels
    :param barge: number of barges
    :return: overall project duration, ocv_s time, ocv_l time, barge time
    """
    # set empty list for respective vessel time
    t_array = []
    t_ocv_s = []
    t_ocv_l = []
    t_barge = []

    # Check if inputs are scalars or lists
    if isinstance(ocv_s, (int, float)) and isinstance(ocv_l, (int, float)) and isinstance(barge, (int, float)):
        # For scalar inputs, we only need one iteration
        ocv_s_i = ocv_s
        ocv_l_i = ocv_l
        barge_i = barge

        # initialize timers and counter
        inf_loop_prevent = 0
        time_ocv_s_i = 0
        time_ocv_l_i = 0
        time_barge_i = 0
        anchor_counter = 0

        # define ship deck space option
        ds_ocv_s = ship_options['OCV small']['deck_space']
        ds_ocv_l = ship_options['OCV big']['deck_space']
        ds_barge = ship_options['Barge']['deck_space']

        # iteration through installation process as long as number of anchor to install is smaller than
        # the number of anchors available
        while n_anchors - anchor_counter > 0:
            # check if ships are fully or only partially loaded
            if n_anchors - anchor_counter < ocv_s_i * ds_ocv_s + ocv_l_i * ds_ocv_l + barge_i * ds_barge:
                n = ocv_s_i + ocv_l_i + barge_i  # Number of vessels
                # number of anchors left is number of remaining anchors divided by number of ships (oversimplification)
                if n > 0:
                    anchors_left_per_vessel = ceil((n_anchors - anchor_counter) / n)
                    diff_1 = 0
                    diff_2 = 0
                    ds_ocv_s = anchors_left_per_vessel
                    ds_ocv_l = anchors_left_per_vessel
                    ds_barge = anchors_left_per_vessel

                    # distribute remaining number of anchors on ship starting with the OCV small
                    # if number of remaining anchors to store is smaller then the deck space of the OCV small
                    # are distributed on to the OCV big
                    if ds_ocv_s > ship_options['OCV small']['deck_space'] and ocv_s_i > 0:
                        diff_1 = ocv_s_i * (anchors_left_per_vessel - ship_options['OCV small']['deck_space'])
                        ds_ocv_s = ship_options['OCV small']['deck_space']

                        # if the deck space of the OCV big is exceeded the remaining anchors are stored on the barge
                        if ocv_l_i != 0:
                            if ds_ocv_l + diff_1 / ocv_l_i > ship_options['OCV big']['deck_space']:
                                diff_2 = ocv_l_i * (
                                        anchors_left_per_vessel + round(diff_1 / ocv_l_i) - ship_options['OCV big'][
                                    'deck_space'])
                                ds_ocv_l = ship_options['OCV big']['deck_space']
                                if barge_i > 0:
                                    ds_barge += diff_2 / barge_i
                            else:
                                ds_ocv_l = anchors_left_per_vessel + ceil(diff_1 / (ocv_l_i + barge_i)) if (ocv_l_i + barge_i) > 0 else 0
                                ds_barge = anchors_left_per_vessel + ceil(diff_1 / (ocv_l_i + barge_i)) if (ocv_l_i + barge_i) > 0 else 0
                        elif barge_i > 0:
                            ds_barge = anchors_left_per_vessel + ceil(diff_1 / barge_i)
                else:
                    # No ships available
                    time_ocv_s_i += 1e4
                    time_ocv_l_i += 1e4
                    time_barge_i += 1e4
                    break

            #  increase respective ship time
            time_ocv_s_i += ocv_s_i * ds_ocv_s * time_installation
            time_ocv_l_i += ocv_l_i * ds_ocv_l * time_installation
            time_barge_i += barge_i * ds_barge * time_installation

            # increase anchor counter to number of installed anchors
            anchor_counter += ocv_s_i * ship_options['OCV small']['deck_space'] + ocv_l_i * ship_options['OCV big'][
                'deck_space'] + barge_i * ship_options['Barge']['deck_space']

            if n_anchors - anchor_counter <= 0:  # check if it is still the case after installation of last anchors
                time_ocv_s_i += ocv_s_i * time_bunkering[0]  # add time to load new anchor
                time_ocv_l_i += ocv_l_i * time_bunkering[1]
                time_barge_i += barge_i * time_bunkering[2]

            inf_loop_prevent += 1  # preventing an infinite loop (if sum of ships is zero)

            # if no anchors are installed, the while loop returns a high value for the timers and breaks the while loop
            if inf_loop_prevent > 20:
                time_ocv_s_i += 1e4
                time_ocv_l_i += 1e4
                time_barge_i += 1e4
                break

        # time is added to overall list of alternatives
        t_ocv_s.append(time_ocv_s_i)
        t_ocv_l.append(time_ocv_l_i)
        t_barge.append(time_barge_i)
        t_array.append(max(time_ocv_s_i, time_ocv_l_i, time_barge_i))

        return t_array[0], t_ocv_s[0], t_ocv_l[0], t_barge[0]
    else:
        # Handle list inputs
        for i in range(len(ocv_s)):
            ocv_s_i = ocv_s[i]
            ocv_l_i = ocv_l[i]
            barge_i = barge[i]

            # The rest of the code for list processing...
            # (similar to above, but for each element in the lists)

        return t_array, t_ocv_s, t_ocv_l, t_barge


In [ ]:
def objective_costs(diameter, length, t_ocv_s, t_ocv_l, t_barge):
    """Function to calculate the installation costs"""
    # Check if inputs are scalars
    if isinstance(diameter, (int, float)) and isinstance(length, (int, float)) and \
       isinstance(t_ocv_s, (int, float)) and isinstance(t_ocv_l, (int, float)) and \
       isinstance(t_barge, (int, float)):
        # Process single scalar inputs
        t = 0.02 * diameter
        mass_steel = (pi * length * diameter * t + pi / 4 * diameter ** 2 * t) * 7.85  # mT
        production_costs_anchor = (mass_steel * 815 + 40000) * n_anchors  # Calculate material cost

        costs_ocv_s = t_ocv_s * ship_options['OCV small']['day_rate']
        costs_ocv_l = t_ocv_l * ship_options['OCV big']['day_rate']
        costs_barge = t_barge * ship_options['Barge']['day_rate']

        return production_costs_anchor + costs_ocv_s + costs_ocv_l + costs_barge
    else:
        # Process list inputs
        # Convert any scalar inputs to lists
        if isinstance(diameter, (int, float)):
            diameter = [diameter] * len(t_ocv_s)
        if isinstance(length, (int, float)):
            length = [length] * len(t_ocv_s)
        if isinstance(t_ocv_s, (int, float)):
            t_ocv_s = [t_ocv_s] * len(diameter)
        if isinstance(t_ocv_l, (int, float)):
            t_ocv_l = [t_ocv_l] * len(diameter)
        if isinstance(t_barge, (int, float)):
            t_barge = [t_barge] * len(diameter)

        result = []

        for i in range(len(diameter)):
            t = 0.02 * diameter[i]
            mass_steel = (pi * length[i] * diameter[i] * t + pi / 4 * diameter[i] ** 2 * t) * 7.85  # mT
            production_costs_anchor = (mass_steel * 815 + 40000) * n_anchors  # Calculate material cost

            costs_ocv_s = t_ocv_s[i] * ship_options['OCV small']['day_rate']
            costs_ocv_l = t_ocv_l[i] * ship_options['OCV big']['day_rate']
            costs_barge = t_barge[i] * ship_options['Barge']['day_rate']

            result.append(production_costs_anchor + costs_ocv_s + costs_ocv_l + costs_barge)

        return result


In [ ]:
def objective_fleet_utilization(ocv_s, ocv_l, barge):
    """Function to calculate the fleet utilization"""
    # Check if inputs are scalars
    if isinstance(ocv_s, (int, float)) and isinstance(ocv_l, (int, float)) and isinstance(barge, (int, float)):
        # Process single scalar inputs
        chance_ocv_s = ship_options['OCV small']['chance'] ** ocv_s
        chance_ocv_l = ship_options['OCV big']['chance'] ** ocv_l
        chance_barge = ship_options['Barge']['chance'] ** barge

        # Calculate combined probability of fleet utilization
        return chance_ocv_s * chance_ocv_l * chance_barge
    else:
        # Process list inputs
        # Convert any scalar inputs to lists
        if isinstance(ocv_s, (int, float)):
            ocv_s = [ocv_s] * len(ocv_l)
        if isinstance(ocv_l, (int, float)):
            ocv_l = [ocv_l] * len(ocv_s)
        if isinstance(barge, (int, float)):
            barge = [barge] * len(ocv_s)

        result = []

        for i in range(len(ocv_s)):
            chance_ocv_s = ship_options['OCV small']['chance'] ** ocv_s[i]
            chance_ocv_l = ship_options['OCV big']['chance'] ** ocv_l[i]
            chance_barge = ship_options['Barge']['chance'] ** barge[i]

            # Calculate combined probability of fleet utilization
            combined_probability = chance_ocv_s * chance_ocv_l * chance_barge
            result.append(combined_probability)

        return result


In [ ]:
def objective_co2(ocv_s, ocv_l, barge, t_ocv_s, t_ocv_l, t_barge):
    """Function to calculate the CO2 emissions"""
    # Check if inputs are scalars
    if isinstance(ocv_s, (int, float)) and isinstance(ocv_l, (int, float)) and \
       isinstance(barge, (int, float)) and isinstance(t_ocv_s, (int, float)) and \
       isinstance(t_ocv_l, (int, float)) and isinstance(t_barge, (int, float)):
        # Process single scalar inputs
        co2_emission_ocv_s = t_ocv_s * ship_options['OCV small']['CO2_emission'] * ocv_s
        co2_emission_ocv_l = t_ocv_l * ship_options['OCV big']['CO2_emission'] * ocv_l
        co2_emission_barge = t_barge * ship_options['Barge']['CO2_emission'] * barge

        return co2_emission_ocv_s + co2_emission_ocv_l + co2_emission_barge
    else:
        # Process list inputs
        # Convert any scalar inputs to lists
        if isinstance(ocv_s, (int, float)):
            ocv_s = [ocv_s] * len(t_ocv_s)
        if isinstance(ocv_l, (int, float)):
            ocv_l = [ocv_l] * len(t_ocv_s)
        if isinstance(barge, (int, float)):
            barge = [barge] * len(t_ocv_s)
        if isinstance(t_ocv_s, (int, float)):
            t_ocv_s = [t_ocv_s] * len(ocv_s)
        if isinstance(t_ocv_l, (int, float)):
            t_ocv_l = [t_ocv_l] * len(ocv_s)
        if isinstance(t_barge, (int, float)):
            t_barge = [t_barge] * len(ocv_s)

        result = []

        for i in range(len(ocv_s)):
            co2_emission_ocv_s = t_ocv_s[i] * ship_options['OCV small']['CO2_emission'] * ocv_s[i]
            co2_emission_ocv_l = t_ocv_l[i] * ship_options['OCV big']['CO2_emission'] * ocv_l[i]
            co2_emission_barge = t_barge[i] * ship_options['Barge']['CO2_emission'] * barge[i]

            result.append(co2_emission_ocv_s + co2_emission_ocv_l + co2_emission_barge)

        return result


In [ ]:
def _solve_ta_ta(p, tension_mudline, theta_m, za, d, mu, su, nc=7.6):
    """Solve the force o the anchor and its angle, based on the tension and angle of the mooring line at the seabed"""
    tension_a, theta = p
    za_q = mooring_data['AWB'] * d * nc * su * za

    return (2 * za_q / tension_a) - (theta ** 2 - theta_m ** 2), np.exp(
        mu * (theta - theta_m)) - tension_mudline / tension_a


In [ ]:
def check_anchor_resistance(diameter, length):
    """Check if the anchor can resist the forces imposed on it"""
    # Check if inputs are scalars
    if isinstance(diameter, (int, float)) and isinstance(length, (int, float)):
        # Process single scalar inputs
        t = 0.02 * diameter
        d_i = diameter - t
        d_e = diameter + t
        mean_diameter = diameter
        weight_anchor = (np.pi * length * mean_diameter * t + np.pi * mean_diameter ** 2 * t / 4) * (
                constants['W_steel'] - constants['W_water'])

        weight_plug = np.pi / 4 * d_i ** 2 * length * soil_data['sat_weight']

        external_shaft_fric = np.pi * d_e * length * soil_data['a_o'] * soil_data['su']
        internal_shaft_fric = np.pi * d_i * length * soil_data['a_i'] * soil_data['su']
        reverse_end_bearing = 6.7 * soil_data['su'] * d_e ** 2 * np.pi / 4

        v_mode_1 = weight_anchor + external_shaft_fric + reverse_end_bearing
        v_mode_2 = weight_anchor + external_shaft_fric + internal_shaft_fric
        v_mode_3 = weight_anchor + external_shaft_fric + weight_plug

        v_max = min(v_mode_1, v_mode_2, v_mode_3)
        h_max = length * d_e * 10 * soil_data['su']

        rel_pos_pad_eye = 0.5

        x = fsolve(_solve_ta_ta, np.array([10000, 1]),
                   (max_t, 0, rel_pos_pad_eye * length, mooring_data['d'], mooring_data['mu'], soil_data['su'], 12))
        tension_pad_eye = x[0]
        angle_pad_eye = x[1]

        h = np.cos(angle_pad_eye) * tension_pad_eye
        v = np.sin(angle_pad_eye) * tension_pad_eye

        a = length / mean_diameter + 0.5
        b = length / (3 * mean_diameter) + 4.5

        hor_util = h / h_max
        ver_util = v / v_max

        # Constraint is (hor_util ** a + ver_util ** b) - 1 <= 0
        # We'll return this value for checking
        return (hor_util ** a + ver_util ** b) - 1
    else:
        # Process list inputs
        # Convert any scalar inputs to lists
        if isinstance(diameter, (int, float)):
            diameter = [diameter] * len(length)
        if isinstance(length, (int, float)):
            length = [length] * len(diameter)

        result = []

        for i in range(len(diameter)):
            t = 0.02 * diameter[i]
            d_i = diameter[i] - t
            d_e = diameter[i] + t
            mean_diameter = diameter[i]
            weight_anchor = (np.pi * length[i] * mean_diameter * t + np.pi * mean_diameter ** 2 * t / 4) * (
                    constants['W_steel'] - constants['W_water'])

            weight_plug = np.pi / 4 * d_i ** 2 * length[i] * soil_data['sat_weight']

            external_shaft_fric = np.pi * d_e * length[i] * soil_data['a_o'] * soil_data['su']
            internal_shaft_fric = np.pi * d_i * length[i] * soil_data['a_i'] * soil_data['su']
            reverse_end_bearing = 6.7 * soil_data['su'] * d_e ** 2 * np.pi / 4

            v_mode_1 = weight_anchor + external_shaft_fric + reverse_end_bearing
            v_mode_2 = weight_anchor + external_shaft_fric + internal_shaft_fric
            v_mode_3 = weight_anchor + external_shaft_fric + weight_plug

            v_max = min(v_mode_1, v_mode_2, v_mode_3)
            h_max = length[i] * d_e * 10 * soil_data['su']

            rel_pos_pad_eye = 0.5

            x = fsolve(_solve_ta_ta, np.array([10000, 1]),
                       (max_t, 0, rel_pos_pad_eye * length[i], mooring_data['d'], mooring_data['mu'], soil_data['su'], 12))
            tension_pad_eye = x[0]
            angle_pad_eye = x[1]

            h = np.cos(angle_pad_eye) * tension_pad_eye
            v = np.sin(angle_pad_eye) * tension_pad_eye

            a = length[i] / mean_diameter + 0.5
            b = length[i] / (3 * mean_diameter) + 4.5

            hor_util = h / h_max
            ver_util = v / v_max

            # Constraint is (hor_util ** a + ver_util ** b) - 1 <= 0
            # We'll return this value for checking
            constraint_value = (hor_util ** a + ver_util ** b) - 1
            result.append(constraint_value)

        return result


In [ ]:
def check_min_vessels(ocv_s, ocv_l, barge):
    """Check if there is at least one vessel on the project"""
    # Check if inputs are scalars
    if isinstance(ocv_s, (int, float)) and isinstance(ocv_l, (int, float)) and isinstance(barge, (int, float)):
        # Process single scalar inputs
        return -1 * (ocv_s + ocv_l + barge) + 1  # Should be <= 0
    else:
        # Process list inputs
        # Convert any scalar inputs to lists
        if isinstance(ocv_s, (int, float)):
            ocv_s = [ocv_s] * len(ocv_l)
        if isinstance(ocv_l, (int, float)):
            ocv_l = [ocv_l] * len(ocv_s)
        if isinstance(barge, (int, float)):
            barge = [barge] * len(ocv_s)

        result = []
        for i in range(len(ocv_s)):
            constraint_value = -1 * (ocv_s[i] + ocv_l[i] + barge[i]) + 1  # Should be <= 0
            result.append(constraint_value)

        return result


In [ ]:
# arrays for plotting continuous preference curves
c1 = np.linspace(45, 113)
c2 = np.linspace(9_500_000, 17_000_000)
c3 = np.linspace(0, 1)
c4 = np.linspace(3_200, 10_200)

# x_points: the outcomes of the objective for which a preference score is defined by the stakeholders
# p_points: the corresponding preference scores
x_points_1, p_points_1 = [[45, 80, 113], [100, 60, 0]]
x_points_2, p_points_2 = [[9_500_000, 11_000_000, 17_000_000], [100, 20, 0]]
x_points_3, p_points_3 = [[0, 0.6, 1], [100, 50, 0]]
x_points_4, p_points_4 = [[3_200, 5_000, 10_200], [100, 40, 0]]


# Stakeholder 1: Project developer (prioritizing cost and project duration)
def s1_preference_project_duration(time):
    """Stakeholder 1's preference for project duration (shorter is better)"""
    if time <= x_points_1[0]:
        return p_points_1[0]  # Best case: 100
    elif time >= x_points_1[-1]:
        return p_points_1[-1]  # Worst case: 0
    else:
        return float(pchip_interpolate(x_points_1, p_points_1, time))

def s1_preference_co2(emissions):
    """Stakeholder 1's preference for CO2 emissions (lower is better)"""
    if emissions <= x_points_4[0]:
        return p_points_4[0]  # Best case: 100
    elif emissions >= x_points_4[-1]:
        return p_points_4[-1]  # Worst case: 0
    else:
        return float(pchip_interpolate(x_points_4, p_points_4, emissions))

def s2_preference_costs(cost):
    """Stakeholder 2's preference for installation costs (lower is better)"""
    if cost <= x_points_2[0]:
        return p_points_2[0]  # Best case: 100
    elif cost >= x_points_2[-1]:
        return p_points_2[-1]  # Worst case: 0
    else:
        return float(pchip_interpolate(x_points_2, p_points_2, cost))

# Stakeholder 2: Marine contractor (prioritizing fleet utilization and CO2 emissions)
def s2_preference_fleet_utilization(util):
    """Stakeholder 2's preference for fleet utilization (higher is better)"""
    if util <= x_points_3[0]:
        return p_points_3[0]  # Worst case: 0
    elif util >= x_points_3[-1]:
        return p_points_3[-1]  # Best case: 100
    else:
        return float(pchip_interpolate(x_points_3, p_points_3, util))


In [ ]:
# Define stakeholder preferences
stakeholder_preferences = {
    1: {
        "a_project_duration": s1_preference_project_duration,
        "b_co2": s1_preference_co2,
    },
    2: {
        "c_costs": s2_preference_costs,
        "d_fleet_utilization": s2_preference_fleet_utilization,
    },
}

# Define objective weights for each stakeholder
objective_weights = {
    1: {"a_project_duration": 0.30, "b_co2": 0.20},
    2: {"c_costs": 0.35, "d_fleet_utilization": 0.15},
}

# Set optimization options
optimization_options = {
    'population_size': 250,
    'generations': 30,
    'mutation_rate': 0.70,
    'hall_of_fame_size': 100,
}


In [ ]:
# Define the main simulation function to be optimized with MODO
@Preferendus(
    preference_functions=stakeholder_preferences,
    stakeholder_weights=[0.5, 0.5],  # Equal weight for both stakeholders
    variable_types=[int, int, int, float, float],
    value_range=[
        (0, ship_options['OCV small']['max_available']),
        (0, ship_options['OCV big']['max_available']),
        (0, ship_options['Barge']['max_available']),
        (1.5, 4.0),
        (2.0, 8.0)
    ],
    objective_weights=objective_weights,
    optimization_options=optimization_options,
    constraints=[
        lambda x: check_min_vessels(x[0], x[1], x[2]) <= 0,
        lambda x: check_anchor_resistance(x[3], x[4]) <= 0
    ]
)
def run_simulation(x):
    """Simulation function for the optimization of floating wind farm installation.
    
    Parameters
    ----------
    x[0]: Number of small OCVs
    x[1]: Number of large OCVs
    x[2]: Number of barges
    x[3]: Diameter of anchor (m)
    x[4]: Length of anchor (m)
    
    Returns
    -------
    - dummy env
    - dummy objects
    - project_duration: Total project duration (days)
    - costs: Total installation costs (Euro)
    - fleet_utilization: Fleet utilization metric (0-1)
    - co2: CO2 emissions (tonnes)

    """
    # Round integer variables to whole numbers (just to be sure)
    ocv_s = int(round(x[0]))
    ocv_l = int(round(x[1]))
    barge = int(round(x[2]))
    diameter = x[3]
    length = x[4]

    # Calculate all objectives
    project_time, time_ocv_s, time_ocv_l, time_barge = objective_time(ocv_s, ocv_l, barge)
    costs = objective_costs(diameter, length, time_ocv_s, time_ocv_l, time_barge)
    fleet_util = objective_fleet_utilization(ocv_s, ocv_l, barge)
    co2_emission = objective_co2(ocv_s, ocv_l, barge, time_ocv_s, time_ocv_l, time_barge)

    # Return required format: env, objects, *objectives
    return None, None, project_time, co2_emission, costs, fleet_util


In [ ]:
# Run the optimization
opt_x = [1, 1, 1, 2.5, 4.0]  # Initial guess for the optimization variables
result = run_simulation(opt_x)  # Initial guess
print(f"Optimal variables: {result['x_opt']}, Utility: {result['utility']}")

In [ ]:
# Evaluate the optimal solution
opt_x = result['x_opt']
ocv_s = int(round(opt_x[0]))
ocv_l = int(round(opt_x[1]))
barge = int(round(opt_x[2]))
diameter = opt_x[3]
length = opt_x[4]

# Calculate objective values for the optimal solution
project_time, time_ocv_s, time_ocv_l, time_barge = objective_time(ocv_s, ocv_l, barge)
costs = objective_costs(diameter, length, time_ocv_s, time_ocv_l, time_barge)
fleet_util = objective_fleet_utilization(ocv_s, ocv_l, barge)
co2_emission = objective_co2(ocv_s, ocv_l, barge, time_ocv_s, time_ocv_l, time_barge)

# Calculate preference scores
pref_s1_duration = s1_preference_project_duration(project_time)
pref_s1_co2 = s1_preference_co2(co2_emission)
pref_s2_costs = s2_preference_costs(costs)
pref_s2_fleet = s2_preference_fleet_utilization(fleet_util)

print("\nDetailed results for optimal solution:")
print(f"Number of small OCVs: {ocv_s}")
print(f"Number of large OCVs: {ocv_l}")
print(f"Number of barges: {barge}")
print(f"Anchor diameter: {diameter:.2f} m")
print(f"Anchor length: {length:.2f} m")
print("\nObjective values:")
print(f"Project duration: {project_time:.2f} days")
print(f"Installation costs: €{costs:,.2f}")
print(f"Fleet utilization: {fleet_util:.3f}")
print(f"CO2 emissions: {co2_emission:.2f} tonnes")
print("\nPreference scores:")
print(f"Stakeholder 1 - Project duration: {pref_s1_duration:.2f}")
print(f"Stakeholder 1 - CO2 emissions: {pref_s1_co2:.2f}")
print(f"Stakeholder 2 - Costs: {pref_s2_costs:.2f}")
print(f"Stakeholder 2 - Fleet utilization: {pref_s2_fleet:.2f}")

In [ ]:
# Visualize the preference functions and optimal solution
fig, axs = plt.subplots(2, 2, figsize=(15, 10))

# Project duration preference
axs[0, 0].plot(c1, pchip_interpolate(x_points_1, p_points_1, c1))
axs[0, 0].scatter(x_points_1, p_points_1, color='blue', s=50)  # Show the actual points used for interpolation
axs[0, 0].scatter([project_time], [pref_s1_duration], color='red', s=100, marker='*')
axs[0, 0].set_xlabel('Project Duration (days)')
axs[0, 0].set_ylabel('Preference')
axs[0, 0].set_title('Project Duration Preference')
axs[0, 0].grid(True)

# Cost preference
axs[0, 1].plot(c2 / 1_000_000, pchip_interpolate(x_points_2, p_points_2, c2))  # Display in millions
axs[0, 1].scatter([x/1_000_000 for x in x_points_2], p_points_2, color='blue', s=50)  # Show interpolation points
axs[0, 1].scatter([costs / 1_000_000], [pref_s2_costs], color='red', s=100, marker='*')
axs[0, 1].set_xlabel('Installation Costs (million €)')
axs[0, 1].set_ylabel('Preference')
axs[0, 1].set_title('Cost Preference')
axs[0, 1].grid(True)

# Fleet utilization preference
axs[1, 0].plot(c3, pchip_interpolate(x_points_3, p_points_3, c3))
axs[1, 0].scatter(x_points_3, p_points_3, color='blue', s=50)  # Show interpolation points
axs[1, 0].scatter([fleet_util], [pref_s2_fleet], color='red', s=100, marker='*')
axs[1, 0].set_xlabel('Fleet Utilization')
axs[1, 0].set_ylabel('Preference')
axs[1, 0].set_title('Fleet Utilization Preference')
axs[1, 0].grid(True)

# CO2 emissions preference
axs[1, 1].plot(c4, pchip_interpolate(x_points_4, p_points_4, c4))
axs[1, 1].scatter(x_points_4, p_points_4, color='blue', s=50)  # Show interpolation points
axs[1, 1].scatter([co2_emission], [pref_s1_co2], color='red', s=100, marker='*')
axs[1, 1].set_xlabel('CO2 Emissions (tonnes)')
axs[1, 1].set_ylabel('Preference')
axs[1, 1].set_title('CO2 Emissions Preference')
axs[1, 1].grid(True)

plt.tight_layout()
plt.show()

